<a href="https://colab.research.google.com/github/Dogbware/Mineria_datos/blob/main/funciones_avanzadas_pandas_p2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Une orders_a y orders_b en un solo DataFrame orders con índice reiniciado.

In [19]:
import pandas as pd
orden_a = pd.read_csv('orders_a.csv')
orden_b = pd.read_csv('orders_b.csv')
clientes = pd.read_csv('customers.csv')
productos = pd.read_csv('products.csv')

orders = pd.concat([orden_a, orden_b])
orders.reset_index(drop=True, inplace=True)
orders.head()

,order_id,order_date,line_id,customer_id,product_id,qty,channel
0,2001,2026-01-02,1,C001,P10,2,web
1,2002,2026-01-02,2,C002,P20,1,store
2,2003,2026-01-03,3,C003,P20,3,web
3,2004,2026-01-04,4,C001,P30,1,app
4,2005,2026-01-05,5,C004,P10,5,web


Ordena orders por order_date ascendente y, en caso de empate, por qty descendente.

In [7]:
agrupados = orders.sort_values(
    by = ['order_date', 'qty'],
    ascending = [True, False]
    )
print(agrupados)

   order_id  order_date  line_id customer_id product_id  qty channel
0      2001  2026-01-02        1        C001        P10    2     web
1      2002  2026-01-02        2        C002        P20    1   store
2      2003  2026-01-03        3        C003        P20    3     web
3      2004  2026-01-04        4        C001        P30    1     app
4      2005  2026-01-05        5        C004        P10    5     web
5      2006  2026-01-05        6        C005        P40    1   store
6      2007  2026-01-06        7        C002        P30    2     web
7      2008  2026-01-07        8        C006        P50    1     app
8      2009  2026-01-08        9        C007        P10    1     web
9      2010  2026-01-09       10        C008        P20    2   store


In [11]:
fecha = (
    orders
    .set_index('order_date')
    .sort_index()
)
print(fecha)

            order_id  line_id customer_id product_id  qty channel
order_date                                                       
2026-01-02      2001        1        C001        P10    2     web
2026-01-02      2002        2        C002        P20    1   store
2026-01-03      2003        3        C003        P20    3     web
2026-01-04      2004        4        C001        P30    1     app
2026-01-05      2005        5        C004        P10    5     web
2026-01-05      2006        6        C005        P40    1   store
2026-01-06      2007        7        C002        P30    2     web
2026-01-07      2008        8        C006        P50    1     app
2026-01-08      2009        9        C007        P10    1     web
2026-01-09      2010       10        C008        P20    2   store


Merge (enriquecer órdenes con clientes):
Haz un merge para añadir a orders las columnas del cliente (customer_name, city, segment) usando customer_id (left join). Valida que sea many_to_one.

In [13]:
orders_cos = pd.merge(
    orders,
    clientes[["customer_id", "customer_name", "city", "segment"]],
    on="customer_id",
    how="left",
    validate="many_to_one"
)
orders_cos.head()

,order_id,order_date,line_id,customer_id,product_id,qty,channel,customer_name,city,segment
0,2001,2026-01-02,1,C001,P10,2,web,Ana,CDMX,Consumer
1,2002,2026-01-02,2,C002,P20,1,store,Luis,GDL,Corporate
2,2003,2026-01-03,3,C003,P20,3,web,Marta,MTY,Consumer
3,2004,2026-01-04,4,C001,P30,1,app,Ana,CDMX,Consumer
4,2005,2026-01-05,5,C004,P10,5,web,Sofia,QRO,Small Biz


Merge (añadir producto y calcular total):
Haz un merge adicional con products para añadir product, category, unit_price y crea una columna line_total = qty * unit_price. Luego ordena por line_total descendente y muestra top 5.

In [23]:
orden_completa = (
    pd.merge(
        orders_cos,
        productos[["product_id", "category", "unit_price"]],
        on="product_id",
        how="left",
        validate="many_to_one"
    )
)

orden_completa["line_total"] = (
    orden_completa["qty"] * orden_completa["unit_price"]
)

orden_completa = orden_completa.sort_values(
    by="line_total",
    ascending=False
)

orden_completa.head(5)


,order_id,order_date,line_id,customer_id,product_id,qty,channel,customer_name,city,segment,category,unit_price,line_total
6,2007,2026-01-06,7,C002,P30,2,web,Luis,GDL,Corporate,Electronics,160,320
5,2006,2026-01-05,6,C005,P40,1,store,Diego,GDL,Consumer,Furniture,220,220
3,2004,2026-01-04,4,C001,P30,1,app,Ana,CDMX,Consumer,Electronics,160,160
2,2003,2026-01-03,3,C003,P20,3,web,Marta,MTY,Consumer,Accessories,45,135
9,2010,2026-01-09,10,C008,P20,2,store,Ramon,CDMX,Small Biz,Accessories,45,90
